# IT Ticket Classifier — Fine-Tuning on Google Colab
Fine-tunes DistilBERT on the `Console-AI/IT-helpdesk-synthetic-tickets` dataset.

**Steps:**
1. Run all cells in order
2. Download `saved_model.zip` at the end
3. Extract into `fine_tuning/saved_model/` in your project
4. Also download `training_results.json` into `fine_tuning/`
5. Push both to GitHub

In [ ]:
# Cell 1: Install packages
!pip install -q transformers datasets accelerate evaluate scikit-learn

In [ ]:
# Cell 2: Dataset setup
import random
from datasets import Dataset, DatasetDict, load_dataset

LABEL2ID = {
    'network_connectivity': 0,
    'software_installation': 1,
    'hardware_malfunction': 2,
    'security_incident': 3,
    'account_access': 4,
    'email_issues': 5,
    'printer_problems': 6,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

CATEGORY_MAP = {
    'Network': 'network_connectivity',
    'RemoteWork': 'network_connectivity',
    'Infrastructure': 'network_connectivity',
    'Software': 'software_installation',
    'Licensing': 'software_installation',
    'Hardware': 'hardware_malfunction',
    'Performance': 'hardware_malfunction',
    'Security': 'security_incident',
    'Account': 'account_access',
    'Communication': 'email_issues',
}

PRINTER_TEMPLATES = [
    'Printer on floor 3 is offline and not printing any jobs. Queue is building up.',
    'Paper is jamming inside the printer every 2-3 pages. Already cleared once.',
    'My print jobs are stuck in the queue and not printing even after restarting.',
    'Printer is printing blank pages. Toner was just replaced.',
    'Cannot find the printer in the network. It was working fine until last week.',
    'Printed document quality is poor with streaks and smudges all over.',
    'Network printer driver installation failed on my new laptop.',
    'Color printer is printing in black and white only even when color is selected.',
    'Print job is being sent to a different printer than the one I selected.',
    'Printer makes loud noise when printing and sometimes does not finish the job.',
    'Duplex printing is not working. It prints single-sided despite my settings.',
    'Scanner on the MFD is not working. Documents sent to email are not arriving.',
    'Printer on second floor needs toner replacement. It is out of paper too.',
    'I cannot install a printer manually. I do not have admin rights to add printers.',
    'Print server is not reachable. All printers on this floor are unavailable.',
    'My default printer keeps changing after every restart.',
    'Printer is online but takes 10+ minutes to print a simple 2-page document.',
    'HP LaserJet on my desk is showing Replace Imaging Drum message.',
    'I need to set up a new printer for a conference room. Who should I contact?',
    'Printed pages have a gray tint across the whole page. Looks like a drum issue.',
]

def build_dataset(printer_samples=20, seed=42):
    raw = load_dataset('Console-AI/IT-helpdesk-synthetic-tickets', split='train')
    texts, labels = [], []
    for row in raw:
        mapped = CATEGORY_MAP.get(row['category'])
        if mapped is None:
            continue
        text = f"{row['subject']}. {row['description']}".strip()
        texts.append(text)
        labels.append(LABEL2ID[mapped])
    # Add printer synthetic
    random.seed(seed)
    printer = (PRINTER_TEMPLATES * 2)[:printer_samples]
    texts.extend(printer)
    labels.extend([LABEL2ID['printer_problems']] * printer_samples)
    # Shuffle and split
    combined = list(zip(texts, labels))
    random.shuffle(combined)
    texts, labels = zip(*combined)
    split = int(0.8 * len(texts))
    return DatasetDict({
        'train': Dataset.from_dict({'text': list(texts[:split]), 'label': list(labels[:split])}),
        'validation': Dataset.from_dict({'text': list(texts[split:]), 'label': list(labels[split:])}),
    })

dataset = build_dataset()
print(f"Train: {len(dataset['train'])} | Validation: {len(dataset['validation'])}")

In [ ]:
# Cell 3: Tokenize
from transformers import AutoTokenizer, DataCollatorWithPadding

BASE_MODEL = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=128, padding=False)

tokenized = dataset.map(tokenize, batched=True, remove_columns=['text'])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print('Tokenization done.')

In [ ]:
# Cell 4: Load model and train
import numpy as np
import evaluate
import json
import transformers
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=len(LABEL2ID), id2label=ID2LABEL, label2id=LABEL2ID
)

accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)['accuracy']
    f1 = f1_metric.compute(predictions=preds, references=labels, average='macro')['f1']
    return {'accuracy': acc, 'f1_macro': f1}

# transformers>=4.46 renamed evaluation_strategy->eval_strategy and tokenizer->processing_class
_ver = tuple(int(x) for x in transformers.__version__.split('.')[:2])
_new_api = _ver >= (4, 46)

strategy_kwargs = (
    {'eval_strategy': 'epoch', 'save_strategy': 'epoch'}
    if _new_api else
    {'evaluation_strategy': 'epoch', 'save_strategy': 'epoch'}
)
tokenizer_kwarg = {'processing_class': tokenizer} if _new_api else {'tokenizer': tokenizer}

args = TrainingArguments(
    output_dir='saved_model',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_steps=10,
    report_to='none',
    **strategy_kwargs,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    **tokenizer_kwarg,
)

train_result = trainer.train()
metrics = trainer.evaluate()
trainer.save_model('saved_model')
tokenizer.save_pretrained('saved_model')

results = {
    'base_model': BASE_MODEL,
    'num_labels': len(LABEL2ID),
    'train_samples': len(dataset['train']),
    'val_samples': len(dataset['validation']),
    'epochs': 3,
    'final_accuracy': round(metrics.get('eval_accuracy', 0), 4),
    'final_f1_macro': round(metrics.get('eval_f1_macro', 0), 4),
    'train_loss': round(train_result.training_loss, 4),
    'train_runtime_s': round(train_result.metrics.get('train_runtime', 0), 1),
    'label2id': LABEL2ID,
    'id2label': {str(k): v for k, v in ID2LABEL.items()},
}
with open('training_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"transformers version: {transformers.__version__}")
print(f"Accuracy: {results['final_accuracy']:.2%}")
print(f"F1 Macro: {results['final_f1_macro']:.2%}")

In [ ]:
# Cell 5: Download files
import shutil
from google.colab import files

shutil.make_archive('saved_model', 'zip', 'saved_model')
files.download('saved_model.zip')
files.download('training_results.json')